# Resolução de Sistemas Lineares: Métodos Básicos

## Introdução
A resolução de sistemas lineares de equações do tipo $A \cdot x = b$ é uma das tarefas mais frequentes na computação científica. Enquanto matrizes completas exigem algoritmos sofisticados de fatoração (como Eliminação de Gauss ou Decomposição LU), o problema se torna trivial se a matriz dos coeficientes $A$ tiver um formato **triangular**.

Neste material, abordaremos os dois algoritmos base formadores de toda a resolução de sistemas: a **Substituição Sucessiva (Forward Substitution)** e a **Substituição Retroativa (Backward Substitution)**.

## Substituição Sucessiva (Forward Substitution)

Este método é utilizado para resolver sistemas lineares cuja matriz de coeficientes é **Triangular Inferior** (geralmente denotada por $L$, do inglês *Lower*). Ou seja, todos os elementos acima da diagonal principal são zeros.

A grande vantagem é que o processo de descoberta das incógnitas ocorre de cima para baixo de maneira natural. A primeira equação possui apenas uma incógnita, que uma vez descoberta, é substituída na segunda equação (agora possuindo apenas uma incógnita não resolvida), e assim por diante.

## Componentes do Algoritmo

- **$L$**: Matriz quadrada triangular inferior ($n \times n$) contendo os coeficientes estruturados de tal modo que $l_{ij} = 0$ para $j > i$.
- **$b$**: Vetor dos termos independentes ($n \times 1$).
- **$x$**: Vetor das incógnitas que buscamos descobrir.

A equação para qualquer valor iterativo $x_i$ obedece a formulação base:

$$x_1 = \frac{b_1}{l_{11}}$$
$$x_i = \frac{b_i - \sum_{j=1}^{i-1} l_{ij} x_j}{l_{ii}} \quad \text{para } i = 2, \dots, n$$

## Características do Método
- **Complexidade Computacional**: É extremamente rápido e eficiente, exigindo aproximadamente $\frac{n^2}{2}$ operações de ponto flutuante, perfazendo algoritmo com custo da ordem matemática $\mathcal{O}(n^2)$.
- **Garantia de Resolução**: O sistema só é matematicamente solucionável se, e somente se, todos os elementos da diagonal principal na matriz $L$ forem **não-nulos** ($l_{ii} \neq 0$). Caso contrário, uma divisão por zero ocorrerá, revelando um sistema indeterminado ou impossível.


### Experimento

Vamos escrever a estrutura matemática e testar o algoritmo para encontrar a resposta do seguinte sistema linear triangular inferior $L \cdot x = b$:

$$\begin{bmatrix} 2 & 0 & 0 \\ 3 & 5 & 0 \\ 1 & -2 & 4 \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix} = \begin{bmatrix} 6 \\ 4 \\ 2 \end{bmatrix}$$

In [2]:
import numpy as np

def substituicao_sucessiva(L, b):
    n = len(b)
    x = np.zeros(n)
    
    for i in range(n):
        # Se o elemento da diagonal for 0, o sistema quebra
        if L[i][i] == 0:
            raise ValueError(f"Divisão por zero detectada na diagonal inferior L[{i}][{i}].")
            
        # Começamos com bb[i] puro e subtraímos a combinação dos termos já descobertos
        soma = 0.0
        for j in range(i):
            soma += L[i][j] * x[j]
            
        x[i] = (b[i] - soma) / L[i][i]
        
    return x

# Definindo a matriz inferior L e o vetor independente b
L = np.array([
    [2.0,  0.0, 0.0],
    [3.0,  5.0, 0.0],
    [1.0, -2.0, 4.0]
])
b = np.array([6.0, 4.0, 2.0])

vetor_x_inf = substituicao_sucessiva(L, b)
print("A solução das incógnitas (x1, x2, x3) é:", vetor_x_inf)


A solução das incógnitas (x1, x2, x3) é: [ 3.   -1.   -0.75]


## Substituição Retroativa (Backward Substitution)

Funciona através da lógica espelhada da solução anterior. Este método é direcionado à solução rápida de sistemas cuja matriz dos coeficientes é **Triangular Superior** (geralmente denotada por $U$, do inglês *Upper*). Nela, todos os números alocados estritamente abaixo da diagonal principal são nulos.

A extração dos valores se dá agora do fundo para o topo (trazendo o termo *retroativo* ou de *trás-para-frente* indicativo do método). A última equação é sempre de 1 incógnita unicamente.

## Componentes do Algoritmo

- **$U$**: Matriz quadrada triangular superior ($n \times n$) obedecendo à formatação $u_{ij} = 0$ para $i > j$.
- **$b$**: Vetor dos termos independentes ($n \times 1$).
- **$x$**: Incógnitas da solução buscada.

A equação avaliativa de dedução ocorre na ordem invertida do plano dos eixos matriciais:

$$x_n = \frac{b_n}{u_{nn}}$$
$$x_i = \frac{b_i - \sum_{j=i+1}^{n} u_{ij} x_j}{u_{ii}} \quad \text{para } i = n-1, \dots, 1$$

## Características do Método
- **Complexidade Computacional**: Também idêntica à forma sucessiva, fechando numa eficiência de $\mathcal{O}(n^2)$ operações numéricas, muito mais tolerável do que os onerosos custos de $\mathcal{O}(n^3)$ empregados por algoritmos que triangulam sistemas não resolvidos.
- **Regra Diagonal**: Fiel à substituição passada, reitera a exigência incontestável de que não existam instâncias nulas em nenhum ponto da diagonal principal ($u_{ii} \neq 0$). Uma matriz cujo determinante possui fator zero não pode transpor resultados algébricos inteiros consistentes.


### Experimento

Realizaremos o experimento algorítmico elaborando a função reversa para equacionar à prova o sistema final $U \cdot x = b$ abaixo descrito:

$$\begin{bmatrix} 3 & 2 & -1 \\ 0 & 4 & 2 \\ 0 & 0 & 5 \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix} = \begin{bmatrix} 4 \\ 10 \\ 15 \end{bmatrix}$$

In [3]:
def substituicao_retroativa(U, b):
    n = len(b)
    x = np.zeros(n)
    
    # Iterando da última linha até a primeira
    for i in range(n-1, -1, -1):
        if U[i][i] == 0:
            raise ValueError(f"Divisão por zero detectada na diagonal superior U[{i}][{i}].")
            
        # Diferente de iterar os vizinhos à esquerda como na matriz L,
        # buscaremos os vizinhos já descobertos à nossa direita
        soma = 0.0
        for j in range(i+1, n):
            soma += U[i][j] * x[j]
            
        x[i] = (b[i] - soma) / U[i][i]
        
    return x

# Definindo a matriz superior U e seu vetor de termo numérico b
U = np.array([
    [3.0, 2.0, -1.0],
    [0.0, 4.0,  2.0],
    [0.0, 0.0,  5.0]
])
b_u = np.array([4.0, 10.0, 15.0])

vetor_x_sup = substituicao_retroativa(U, b_u)
print("A solução das incógnitas (x1, x2, x3) é:", vetor_x_sup)


A solução das incógnitas (x1, x2, x3) é: [1.66666667 1.         3.        ]


## Eliminação de Gauss (Gaussian Elimination)

O método de **Eliminação de Gauss** consiste em manipular as equações do sistema linear $A \cdot x = b$ para transformá-lo em um sistema equivalente $U \cdot x = b'$ onde a matriz de coeficientes $U$ é uma matriz triangular superior. Uma vez que o sistema encontra-se na forma triangular, sua resolução se torna trivial aplicando o algoritmo já estudado de **Substituição Retroativa (Backward Substitution)**.

A manipulação se dá pelas propriedades matemáticas de matrizes, em que uma linha inteira pode ser multiplicada por uma constante e somada a outra sem alterar a igualdade estrutural do sistema.

## Componentes do Algoritmo

- **$A$**: Matriz quadrada dos coeficientes ($n \times n$).
- **$b$**: Vetor dos termos independentes ($n \times 1$).
- **Multiplicador $m_{ij}$**: Razão utilizada para zerar os elementos abaixo da diagonal, calculada como $m_{ij} = \frac{a_{ij}}{a_{jj}}$, onde $a_{jj}$ é o **pivô** no passo atual.
- **Operação Padrão em Linha**: A linha $L_i$ será substituída por $L_i - m_{ij} L_j$, aplicável tanto na matriz $A$ quanto no vetor $b$, para escalonar o elemento $a_{ij}$ até zero.

## Características do Método
- **Complexidade Computacional**: É o primeiro método direto cujo processamento se torna denso e oneroso para hardwares restritos, pois a fase que converte um sistema completo em um sistema superior requer múltiplas iterações que perfazem aproximadamente $\frac{2n^3}{3}$ operações numéricas, caracterizando uma complexidade $\mathcal{O}(n^3)$.
- **Regra Pivoteamento e Consistência**: O escalonamento na sua forma mais engessada (chamada sem pivoteamento) pressupõe que nenhum coeficiente na diagonal principal resulte em zero ao longo dos passos, pois cada um deles servirá como denominador da operação de razão ($a_{jj} \neq 0$). Caso surja um elemento zero natural no caminho, o algoritmo irá quebrar tentando cometer divisão por zero; tal perigo exige que implementações mais modernas apliquem permutação de linhas quando tal cenário é encontrado.


### Experimento

Iremos construir o formato matricial linear para submeter ao algoritmo de Eliminação de Gauss, forçando-o a triangular até a matriz $U$ superior o seguinte sistema base de exemplo $A \cdot x = b$:

$$\begin{bmatrix} 2 & 1 & -1 \\ -3 & -1 & 2 \\ -2 & 1 & 2 \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix} = \begin{bmatrix} 8 \\ -11 \\ -3 \end{bmatrix}$$

In [4]:
import numpy as np

def eliminacao_gauss(A, b):
    # Criamos cópias seguras para não modificar a original
    M = np.copy(A).astype(float)
    v = np.copy(b).astype(float)
    n = len(v)
    
    # Percorrer cada coluna como pivô
    for j in range(n - 1):
        if M[j][j] == 0:
            raise ValueError(f"Pivô M[{j}][{j}] igual a zero encontrado. Solução por método ingênuo interrompida.")
            
        # Percorrer as linhas abaixo da diagonal para o pivô selecionado
        for i in range(j + 1, n):
            # Calcular o multiplicador para zerar o item (i, j)
            m = M[i][j] / M[j][j]
            
            # Subtrair da linha inteira a combinação escalar
            for k in range(j, n):
                M[i][k] = M[i][k] - m * M[j][k]
                
            # Aplicar a mesma operação no termo de igualdade b correspondente
            v[i] = v[i] - m * v[j]
            
    return M, v

# Declarando a matriz base completa
A_gauss = np.array([
    [ 2.0,  1.0, -1.0],
    [-3.0, -1.0,  2.0],
    [-2.0,  1.0,  2.0]
])

b_gauss = np.array([8.0, -11.0, -3.0])

print("Matriz Original A:")
print(A_gauss)

# Passo 1: Transformar matriz A completa para matriz U superior através da Eliminação
U_triangular, b_novo = eliminacao_gauss(A_gauss, b_gauss)

print("\nMatriz Transformada U (Superior):")
print(U_triangular)
print("\nVetor modificado de termos (b'):", b_novo)

# Passo 2: Executar a Substituição Retroativa que já definimos na célula anterior
vetor_x_gauss = substituicao_retroativa(U_triangular, b_novo)

print("\nA solução final das incógnitas (x1, x2, x3) é:", vetor_x_gauss)


Matriz Original A:
[[ 2.  1. -1.]
 [-3. -1.  2.]
 [-2.  1.  2.]]

Matriz Transformada U (Superior):
[[ 2.   1.  -1. ]
 [ 0.   0.5  0.5]
 [ 0.   0.  -1. ]]

Vetor modificado de termos (b'): [8. 1. 1.]

A solução final das incógnitas (x1, x2, x3) é: [ 2.  3. -1.]


## Eliminação de Gauss com Pivotamento Parcial

A versão ingênua da Eliminação de Gauss sofre de uma falha crítica estatística e computacional: se o elemento que atua como pivô atual (o denominador da base algébrica de eliminação) for $0$ (zero), o algoritmo sofre uma divisão por zero e falha imediatamente. Além disso, se o pivô for muito pequeno em módulo mas não zero, a divisão pode gerar multiplicadores excessivamente grandes, mitigando a ampliação de erros intrínsecos de arrendondamento numérico, o que arruína a precisão algébrica em cascata.

O **Pivotamento Parcial** resolve esse infortúnio vital procurando, na coluna do pivô em exploração, pelo componente de **maior valor absoluto** que se encontra na restrita margem que orbita abaixo da diagonal principal em diante, permutando (trocando) a linha atual perniciosa com a linha redentora de maior valor de base recuada, antes de começar os cálculos destrinchativos que irão zerar tal respectiva coluna.

## Componentes Adicionados ao Algoritmo

- **Busca por Posição P**: No passo $j$, procuramos a linha $p$ (comprada em subconjunto $p \ge j$) tal que garanta a maior base $|a_{pj}| = \max_{i \ge j} |a_{ij}|$.
- **Swap (Permutação) Parcial**: Trocamos simultaneamente a linha $j$ fragilizada pela linha $p$ de peso resgatada na matriz $A$ ($L_j \leftrightarrow L_p$) e transladamos os respectivos marcadores $j$ e $p$ no vetor dos termos independentes $b$.

## Características do Método
- **Estabilidade Numérica**: Exerce excepcional resistência a erros flutuantes ao segurar propositalmente todos os multiplicadores de fração mantidos em $m_{ij} \le 1$.
- **Complexidade**: A vasculha do melhor pivô em cada coluna infunde um micro-custo condicional de aproximadamente $\mathcal{O}(n^2)$ num macro-ciclo de $\mathcal{O}(n^3)$, não ferindo a sua performance assintótica generalista e ratificando consistência frente ao custo virtual nulo.


### Experimento - Pivotamento Parcial

Vamos utilizar um sistema singular proposital injetando zero inicial que quebraria a tentativa ingênua, porém perfeitamente absorvível com matriz reversa no primeiro passo do Pivotamento Parcial:

$$\begin{bmatrix} 0 & 2 & 3 \\ 1 & -1 & 1 \\ -1 & 1 & 0 \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix} = \begin{bmatrix} 8 \\ -1 \\ 2 \end{bmatrix}$$

In [5]:
def eliminacao_gauss_parcial(A, b):
    M = np.copy(A).astype(float)
    v = np.copy(b).astype(float)
    n = len(v)
    
    for j in range(n - 1):
        # --- PIVOTAMENTO PARCIAL ---
        # 1. Encontrar a linha com o maior valor absoluto na coluna j, da diagonal para baixo
        pivo_linha = j
        maior_valor = abs(M[j][j])
        
        for k in range(j + 1, n):
            if abs(M[k][j]) > maior_valor:
                maior_valor = abs(M[k][j])
                pivo_linha = k
                
        # 2. Permutar (trocar) a linha atual 'j' pela linha do maior pivô 'pivo_linha'
        if pivo_linha != j:
            # Troca dinâmica simultânea de linhas inteiras
            M[[j, pivo_linha]] = M[[pivo_linha, j]]
            v[[j, pivo_linha]] = v[[pivo_linha, j]]
            
        # 3. Verificação de Matriz Impossível Pós-Pivotamento
        if M[j][j] == 0:
            raise ValueError("Sistema singular ou impossível. Mesmo após o pivotamento, o pivô alcançado é zero.")
            
        # --- ELIMINAÇÃO INALTERADA ---
        for i in range(j + 1, n):
            m = M[i][j] / M[j][j]
            for k in range(j, n):
                M[i][k] = M[i][k] - m * M[j][k]
            v[i] = v[i] - m * v[j]
            
    return M, v

# Matriz e vetor problema
A_parcial = np.array([
    [ 0.0,  2.0, 3.0],
    [ 1.0, -1.0, 1.0],
    [-1.0,  1.0, 0.0]
])
b_parcial = np.array([8.0, -1.0, 2.0])

print("Matriz Original A:")
print(A_parcial)

U_parcial, b_parcial_novo = eliminacao_gauss_parcial(A_parcial, b_parcial)

print("\nMatriz Transformada U (com swap dinâmico seguro de linhas):")
print(U_parcial)

x_parcial = substituicao_retroativa(U_parcial, b_parcial_novo)
print("\nA solução final das incógnitas (x1, x2, x3) é:", x_parcial)


Matriz Original A:
[[ 0.  2.  3.]
 [ 1. -1.  1.]
 [-1.  1.  0.]]

Matriz Transformada U (com swap dinâmico seguro de linhas):
[[ 1. -1.  1.]
 [ 0.  2.  3.]
 [ 0.  0.  1.]]

A solução final das incógnitas (x1, x2, x3) é: [0.5 2.5 1. ]


## Eliminação de Gauss com Pivotamento Total

Enquanto o pivotamento parcial busca estabilidade permutando inteiramente as linhas baseado na análise de uma única coluna pontual, o **Pivotamento Total** (Complete Pivoting) opera com um filtro de precisão cega: ele impõe a busca ostensiva varrendo o **maior membro absoluto de todo o quadrado residual interativo**, vasculhando em tempo real tudo que engloba de forma global restrita nas linhas e colunas remanescentes ainda não desconstruídas.

As trocas necessárias para trazer o elemento global gigante resgatado de onde pertencia da ponta, diretamente para a mira diagonal imperativa atual $a_{jj}$ envolvem necessariamente **trocas simultâneas de sub-eixos em linha *E* coluna**.

## Componentes Adicionados ao Algoritmo

- **Busca Global Integral**: Achar coordenadas engenhadas de intercepto $p$ (linha) e $q$ (coluna) em toda porção sub-espacial tal que $|a_{pq}| = \max_{i \ge j, k \ge j} |a_{ik}|$.
- **Swap Parcial Duplo**:
  1. *Permutação das Linhas*: $L_j \leftrightarrow L_p$ em uníssono espelhado com o acerto lateral $b_j \leftrightarrow b_p$.
  2. *Permutação das Colunas*: $C_j \leftrightarrow C_q$. **Aviso crítico**: Diferente da ordem em linhas que permuta reações aditivas, uma troca de colunas destrói o arranjo original correspondente de cada incógnita indexial do vetor $x$, forçando a implementação a acorrentar numa memória histórica a transposição, garantindo que desfaça a inversão na última estância após resolução triangular.

## Características do Método
- **Garantia Empírica Indissolúvel**: Propicia a margem laboratorial invicta com rigorosidade minuciosa blindando equações caóticas do pior índice de erros flutuantes imaginável na matemática discreta.
- **Custo Superaditivio de Estrutura**: A procura engessante eleva radicalmente a leitura exaustiva do compilador (em ordem $\mathcal{O}(n^3)$ só pras tomadas de rotas em arrays não limítrofes). Por causa da deficiência de custo-benefício computacional gerado pelas quebras de ciclo (que custa horrores de performance L1 e L2), o método fica relegado exclusivamente aos casos extremos engenhados onde o viés Parcial ainda se prove trágico.


### Experimento - Pivotamento Total

Construiremos um esqueleto experimental base propenso a variações dimensionais onde as trocas duplas trarão o elemento massivo cruzado do extremo oposto perfeitamente pro controle pivô central.

$$\begin{bmatrix} 1 & 1 & 1 \\ 1 & 1 & 2 \\ 2 & -1 & 1 \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix} = \begin{bmatrix} 4 \\ 6 \\ 3 \end{bmatrix}$$

In [6]:
def eliminacao_gauss_total(A, b):
    M = np.copy(A).astype(float)
    v = np.copy(b).astype(float)
    n = len(v)
    
    # Vetor para registrar a ordem historica reversível das variáveis já que as colunas mudam 
    # ordem_x começa como [0, 1, 2, ..., n-1] indexando passivamente
    ordem_x = np.arange(n)
    
    for j in range(n - 1):
        # --- PIVOTAMENTO TOTAL ---
        # 1. Encontrar o maior elemento absoluto EM TODA A SUBMATRIZ ATIVA residual M[j:n, j:n]
        pivo_linha = j
        pivo_coluna = j
        maior_valor = 0.0
        
        for i in range(j, n):
            for k in range(j, n):
                if abs(M[i][k]) > maior_valor:
                    maior_valor = abs(M[i][k])
                    pivo_linha = i
                    pivo_coluna = k
                    
        # 2. Permutação das Linhas
        if pivo_linha != j:
            M[[j, pivo_linha]] = M[[pivo_linha, j]]
            v[[j, pivo_linha]] = v[[pivo_linha, j]]
            
        # 3. Permutação das Colunas
        if pivo_coluna != j:
            M[:, [j, pivo_coluna]] = M[:, [pivo_coluna, j]]
            
            # Coração central: Registrar a inversão na memória de identidades (a incógnita c1 vira c2 e vice-versa)
            ordem_x[[j, pivo_coluna]] = ordem_x[[pivo_coluna, j]]
            
        if M[j][j] == 0:
            raise ValueError("Sistema em impasse singular, impossível solucionar sem redundâncias ou sem limites nulos.")
            
        # --- ELIMINAÇÃO ROTINEIRA ---
        for i in range(j + 1, n):
            m = M[i][j] / M[j][j]
            for k in range(j, n):
                M[i][k] = M[i][k] - m * M[j][k]
            v[i] = v[i] - m * v[j]
            
    return M, v, ordem_x

# Matriz de teste laboratorial engenhada
A_total = np.array([
    [ 1.0,  1.0, 1.0],
    [ 1.0,  1.0, 2.0],
    [ 2.0, -1.0, 1.0]
])
b_total = np.array([4.0, 6.0, 3.0])

print("Matriz Original A:")
print(A_total)

U_total, b_total_novo, mapa_incognitas = eliminacao_gauss_total(A_total, b_total)

print("\nMatriz Transformada U (com swap de colunas e linhas):")
print(U_total)
print("Vetor novo de valores (b'): ", b_total_novo)

# Executando a resposta com colunas/valores desalinhados
x_baguncado = substituicao_retroativa(U_total, b_total_novo)

# Fase Final Lógica: Reordenar x final de acordo com a memoria de mapa_incognitas recuperada
x_final_verdadeiro = np.zeros(len(x_baguncado))
for ind, ref_original in enumerate(mapa_incognitas):
    x_final_verdadeiro[ref_original] = x_baguncado[ind]

print(f"\nMapa Histórico de Trocas Identitárias das Colunas: {mapa_incognitas}")
print("\nA solução final correta ordenada e devolvida das incógnitas reais (x1, x2, x3) é:", x_final_verdadeiro)


Matriz Original A:
[[ 1.  1.  1.]
 [ 1.  1.  2.]
 [ 2. -1.  1.]]

Matriz Transformada U (com swap de colunas e linhas):
[[ 2.   1.   1. ]
 [ 0.  -1.5  1.5]
 [ 0.   0.   1. ]]
Vetor novo de valores (b'):  [6. 0. 1.]

Mapa Histórico de Trocas Identitárias das Colunas: [2 1 0]

A solução final correta ordenada e devolvida das incógnitas reais (x1, x2, x3) é: [1. 1. 2.]


## Fatoração LU (Sem Pivotamento)

A eliminação de Gauss convencional não arquiva o seu próprio rastro de operações interativas, impossibilitando com que um mesmo sistema de matriz $A$ seja resolvido no futuro para múltiplos vetores $b$ sem ter que pagar repetidamente o pesado custo cúbico $\mathcal{O}(n^3)$ da desconstrução estrutural.

A **Fatoração LU** explora essa métrica dividindo a matriz original $A$ estaticamente em duas fatias perfeitamente retentoras: $A = L \cdot U$.

- **$L$ (Lower)**: Uma matriz triangular inferior onde a diagonal possui elementos $1$, e os demais elementos albergam cirurgicamente os recíprocos dos multiplicadores $m_{ij}$ achados em cada respectiva etapa de zeragem de Gauss.
- **$U$ (Upper)**: A exata matriz triangular superior idêntica ao que sobra ao final do método de Gauss convencional.

## Resolução do Sistema
Graças à identidade $A = L \cdot U$, um sistema complexo $A \cdot x = b$ é reinterpretado na cadeia em dupla execução de forma trivial ($L \cdot U \cdot x = b$):
1. Adotamos o núcleo abstrato de que $U \cdot x$ nos resulta num vetor oculto $y$. Resolvemos primeiro essa etapa achando a intermediação $y$ equacionando **$L \cdot y = b$** (Substituição Sucessiva rápida).
2. Conhecendo o $y$ transitório, extraímos as raízes reais resolvendo **$U \cdot x = y$** (Substituição Retroativa rápida).

Por resolver as equações matriciais apenas pelas pontas de métodos já mastigados nas $\mathcal{O}(n^2)$, o processamento de $b$'s dinâmicos em laboratório ou indústrias fica escalonável sem engarrafamento.

### Experimento - Fatoração LU

Validaremos o arquétipo em um simples sistema regular:

$$A = \begin{bmatrix} 3 & 2 & 4 \\ 1 & 1 & 2 \\ 4 & 3 & -2 \end{bmatrix}, \quad b = \begin{bmatrix} 1 \\ 2 \\ 3 \end{bmatrix}$$

In [7]:
def fatoracao_lu(A):
    n = len(A)
    U = np.copy(A).astype(float)
    L = np.eye(n) # Matriz Identidade armazena 1's em toda diagonal principal
    
    for j in range(n - 1):
        if U[j][j] == 0:
            raise ValueError("O pivô nativo é zero e o método LU puro falhou.")
            
        for i in range(j + 1, n):
            m = U[i][j] / U[j][j]
            
            # Aloca o multiplicador na posição correlata esvaziada em L
            L[i][j] = m
            
            # Efetua as remoções habituais projetivas exclusivas em U
            for k in range(j, n):
                U[i][k] = U[i][k] - m * U[j][k]
                
    return L, U

def resolver_lu(L, U, b):
    # 1º Passo: Substituição Sucessiva (L*y = b)
    y = substituicao_sucessiva(L, b)
    
    # 2º Passo: Substituição Retroativa (U*x = y)
    x = substituicao_retroativa(U, y)
    return x

# Matrizes de Exemplo para LU
A_lu = np.array([
    [3.0, 2.0,  4.0],
    [1.0, 1.0,  2.0],
    [4.0, 3.0, -2.0]
])
b_lu = np.array([1.0, 2.0, 3.0])

L_fator, U_fator = fatoracao_lu(A_lu)

print("Matriz L (Inferior - Multiplicadores):")
print(L_fator)
print("\nMatriz U (Superior - Escalonada):")
print(U_fator)

x_lu_resultado = resolver_lu(L_fator, U_fator, b_lu)
print("\nA solução [x1, x2, x3] extraída pelas extremidades das matrizes partidas é:")
print(x_lu_resultado)


Matriz L (Inferior - Multiplicadores):
[[1.         0.         0.        ]
 [0.33333333 1.         0.        ]
 [1.33333333 1.         1.        ]]

Matriz U (Superior - Escalonada):
[[ 3.00000000e+00  2.00000000e+00  4.00000000e+00]
 [ 0.00000000e+00  3.33333333e-01  6.66666667e-01]
 [ 0.00000000e+00  5.55111512e-17 -8.00000000e+00]]

A solução [x1, x2, x3] extraída pelas extremidades das matrizes partidas é:
[-3.00000000e+00  5.00000000e+00  5.55111512e-17]


In [8]:
fatoracao_lu(np.array([[1., 1., 1.],
                      [2., 1., -1.],
                      [3., 2., 0]]))

(array([[1., 0., 0.],
        [2., 1., 0.],
        [3., 1., 1.]]),
 array([[ 1.,  1.,  1.],
        [ 0., -1., -3.],
        [ 0.,  0.,  0.]]))

## Fatoração LU com Pivotamento Parcial

Incorporar a vital técnica de Pivotamento é imperativo diante de inconsistências na matriz e deficiências de erros contínuos (round-offs). Contudo, permutar linhas de forma estocástica arruinaria a organização arquivada estática dos coeficientes atrelados ao vetor na matriz espelho $L$!

Para contornar o engessamento mecânico de variáveis, externalizamos o rastreamento das permutações ao longo do tempo por meio da **Matriz de Permutação $P$**. Essa transdutor age como a malha referencial da reorganização, resultando assim na equação faturada perfeita: **$P \cdot A = L \cdot U$**.

## Algoritmos em Etapas Equacionais Modificadas
Adotando a transfiguração identitária de que a estrutura matricial submetida é agora o termo equacional completo ditado na igualdade $P \cdot A \cdot x = P \cdot b$, substituímos o núcleo esquerdo faturizado isolando a matriz embutida e convertemos perfeitamente para: **$L \cdot U \cdot x = P \cdot b$**.

- **Passo 1 ($P \cdot b$)**: O vetor inicial isolado sofre o produto vetorial pela identidade mapeada permutada $P$, o que tem a consequência de embaralhar as linhas do vetor $b$ para entrarem na ordem cirurgicamente precisa no qual os vetores matriciais de matriz simétrica L esperam ingerir.
- **Passo 2 ($L \cdot y = Pb$)**: Calculado rapidamente no limiar do vetor modificado por Substituição Sucessiva inferior paralela.
- **Passo 3 ($U \cdot x = y$)**: Recuperação do arranjo real incólume por espelhamento transversal superior utilizando a Retroatividade.

### Experimento - LU com Matriz de Permutação P

Vamos processar uma matriz restrita em simulação que necessita forçosamente da permeabilidade matricial no intermédio para quebrar limites assintóticos.

$$A = \begin{bmatrix} 2 & 1 & 1 \\ 4 & -6 & 0 \\ -2 & 7 & 2 \end{bmatrix}, \quad b = \begin{bmatrix} 5 \\ -2 \\ 9 \end{bmatrix}$$

In [9]:
def fatoracao_lu_pivotamento(A):
    n = len(A)
    U = np.copy(A).astype(float)
    L = np.zeros((n, n))
    P = np.eye(n) # A Matriz Identitária base que arquivará espelhamentos P
    
    for j in range(n - 1):
        # Procurar o melhor pivô na coluna iterativa atual do ponto inferior em diante
        pivo_linha = np.argmax(abs(U[j:n, j])) + j
        
        # Efetuar Swap geral de acompanhamento unificado para a matriz
        if pivo_linha != j:
            U[[j, pivo_linha]] = U[[pivo_linha, j]]
            P[[j, pivo_linha]] = P[[pivo_linha, j]]
            L[[j, pivo_linha]] = L[[pivo_linha, j]]   # Mantemos as correções prévias atreladas sob controle dimensional seguro
            
        if U[j][j] == 0:
            raise ValueError("Sistema detectado como singular.")
            
        # Fatoração padrão alimentando L no vácuo e removendo bases residuais
        for i in range(j + 1, n):
            m = U[i][j] / U[j][j]
            L[i][j] = m
            # U[i, j:] é submetido em array slicing numérico ultra rápido pra zerar tudo posterior
            U[i, j:] = U[i, j:] - m * U[j, j:]
            
    # Forzamos o diag 1 integral porque é identidade sub-adjunta
    L = L + np.eye(n)
            
    return L, U, P

def resolver_lu_pivotamento(L, U, P, b):
    # Permutar o vetor 'b' usando produto vetorial puro da arquitetura simétrica da matriz identitária P
    # P @ b na sintaxe compacta é equivalente a numpy.dot(P, b)
    Pb = P @ b
    
    y = substituicao_sucessiva(L, Pb)
    x = substituicao_retroativa(U, y)
    
    return x

A_piv = np.array([
    [ 2.0,  1.0,  1.0],
    [ 4.0, -6.0,  0.0],
    [-2.0,  7.0,  2.0]
])
b_piv = np.array([5.0, -2.0, 9.0])

L_f, U_f, P_permut = fatoracao_lu_pivotamento(A_piv)

print("Matriz de Permutação P (O rastreador de trocas):")
print(P_permut)

print("\nVerificando revalidação da matriz inteira [P * A]:")
print(P_permut @ A_piv)
print("\nVerificando validação cruzada do fator derivado [L * U]:")
print(L_f @ U_f)

x_piv_resultado = resolver_lu_pivotamento(L_f, U_f, P_permut, b_piv)
print("\nA solução final do sistema particionado [x1, x2, x3]:")
print(x_piv_resultado)


Matriz de Permutação P (O rastreador de trocas):
[[0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]

Verificando revalidação da matriz inteira [P * A]:
[[ 4. -6.  0.]
 [ 2.  1.  1.]
 [-2.  7.  2.]]

Verificando validação cruzada do fator derivado [L * U]:
[[ 4. -6.  0.]
 [ 2.  1.  1.]
 [-2.  7.  2.]]

A solução final do sistema particionado [x1, x2, x3]:
[1. 1. 2.]


In [ ]:
import numpy as np

def algoritmo_thomas(A, d):
    n = len(d)
    
    # 1. Extração automática das diagonais
    a = np.zeros(n)
    c = np.zeros(n)
    a[1:] = np.diag(A, k=-1) # Subdiagonal (abaixo da principal)
    b = np.diag(A, k=0)      # Diagonal principal
    c[:-1] = np.diag(A, k=1) # Superdiagonal (acima da principal)
    
    # 2. Inicialização dos vetores de trabalho
    c_linha = np.zeros(n)
    d_linha = np.zeros(n)
    x = np.zeros(n)
    
    if b[0] == 0:
        raise ValueError("Pivô inicial nulo. O algoritmo de Thomas falhou.")

    # 3. Substituição Progressiva
    c_linha[0] = c[0] / b[0]
    d_linha[0] = d[0] / b[0]
    
    for i in range(1, n):
        m = b[i] - a[i] * c_linha[i-1]
        
        if m == 0:
            raise ValueError(f"Pivô nulo encontrado na linha {i}.")
            
        if i < n - 1:
            c_linha[i] = c[i] / m
            
        d_linha[i] = (d[i] - a[i] * d_linha[i-1]) / m
        
    # 4. Substituição Retroativa
    x[n-1] = d_linha[n-1]
    for i in range(n-2, -1, -1):
        x[i] = d_linha[i] - c_linha[i] * x[i+1]
        
    return x

A = np.array([
    [ 2.0, -1.0,  0.0,  0.0],
    [-1.0,  2.0, -1.0,  0.0],
    [ 0.0, -1.0,  2.0, -1.0],
    [ 0.0,  0.0, -1.0,  2.0]
])
d = np.array([1.0, 0.0, 0.0, 1.0])

solucao = algoritmo_thomas(A, d)
print("\nA solução final das incógnitas é:", solucao)

In [ ]:
def criterio_linhas(A):
    n = len(A)
    for i in range(0, n):
        soma_fora_diagonal = sum(abs(A[i][j]) for j in range(n) if j != i)
        if soma_fora_diagonal / abs(A[i][i]) >= 1:
            return False
    return True


def gauss_jacobi(A, b, x0, tol=0.05, max_iter=200):
    if not criterio_linhas(A): 
        print("Matriz não satisfaz os critérios de convergência. Gauss-Jacobi pode não convergir")
    
    x_k = x0
    k = 0

    U = np.triu(A, k=1)
    L = np.tril(A, k=-1)
    D_inv = np.eye(len(A)) / np.diag(A)

    G = -D_inv @ (L + U)
    c = D_inv @ b

    while k < max_iter:
        x_k_new = G @ x_k + c
        if ((np.linalg.norm(x_k_new - x_k, np.inf) / np.linalg.norm(x_k_new, np.inf)) < tol):
            return x_k_new, k+1, (A @ x_k_new)
        
        x_k = x_k_new
        k += 1
    
    return x_k, k, (A @ x_k)

A = np.array([[10., 2., 1.],
              [1., 5., 1.],
              [2., 3., 10.]])

b = np.array([7., -8., 6.])
gauss_jacobi(A, b, np.array([0.7, -1.6, 0.6]))

In [ ]:
def criterio_sassesfeld(A):
    n = len(A)
    beta = []
    for i in range(n):
        somatorio_antes = sum(abs(A[i][j]) * beta[j] for j in range(i))
        somatorio_depois = sum(abs(A[i][j]) for j in range(i+1, n))

        beta_i = (somatorio_antes + somatorio_depois) / abs(A[i][i])
        beta.append(beta_i)

        if beta_i >= 1:
            return False
    return True

def substituicao_sucessiva(L, b):
    n = len(b)
    x = np.zeros(n)
    
    for i in range(n):
        # Se o elemento da diagonal for 0, o sistema quebra
        if L[i][i] == 0:
            raise ValueError(f"Divisão por zero detectada na diagonal inferior L[{i}][{i}].")
            
        # Começamos com bb[i] puro e subtraímos a combinação dos termos já descobertos
        soma = 0.0
        for j in range(i):
            soma += L[i][j] * x[j]
            
        x[i] = (b[i] - soma) / L[i][i]
        
    return x

def gauss_seidel(A, b, x0, tol=0.05, max_iter=200):
    if not criterio_linhas(A) and not criterio_sassesfeld(A):
        print("Matriz não satisfaz os critérios de convergência. Gauss-Seidel pode não convergir")
    
    x_k = x0
    k = 0

    L_D = np.tril(A)
    U = np.triu(A, k=1)

    while k < max_iter:
        x_k_new = substituicao_sucessiva(L_D, -U @ x_k + b)
        if ((np.linalg.norm(x_k_new - x_k, np.inf) / np.linalg.norm(x_k_new, np.inf)) < tol):
            return x_k_new, k+1, (A @ x_k_new)
    
        x_k = x_k_new
        k += 1
    
    return x_k, k, (A @ x_k)

gauss_seidel(A, b, np.array([0.7, -1.6, 0.6]))